In [329]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import pandas as pd
import spacy
import re
import time
import numpy as np
import time

import ast

In [330]:
from spacy.matcher import PhraseMatcher
from spacy.matcher import Matcher
import re

In [331]:
# Model data
model = {'en': spacy.load("en_core_web_sm"), 'de': spacy.load("de_core_news_sm")}

In [332]:
#function to split string in the row by " | "
def split_form(row):
    row_split = row.split(" | ")
    return row_split

In [333]:
#function to split words and sentences into two different dataframes
def split_words_sentences2(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []

    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_alternatives_word.append(row["Alt_split"])
        
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [334]:
#function to split words and sentences into two different dataframes for singular/plural dataframes
def split_words_sentences(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []
    
    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []
    
    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])
        
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_sg_all_sentence, list_pl_sentence, list_pl_all_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [335]:
# read csv file
df_rules = pd.read_csv("rules_08MAR_English_h.csv")

In [336]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1450 entries, 0 to 1449
Data columns (total 15 columns):
Lemma                     1450 non-null object
Word_Type                 1450 non-null object
Priority                  798 non-null object
Category                  1450 non-null object
Primary_subcategory       1450 non-null object
Secondary_subcategory     347 non-null object
Alt_Field                 480 non-null object
Alt_Sg_Replacement        711 non-null object
Alt_Sg_people_first       158 non-null object
Alt_Sg_reframed           298 non-null object
Alt_Pl_collective_noun    47 non-null object
Alt_Pl                    139 non-null object
Alt_Pl_people_first       28 non-null object
Alt_Pl_reframed           4 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 170.0+ KB


### Drop duplicates

In [337]:
df_rules = df_rules.drop_duplicates()

In [338]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1449 entries, 0 to 1449
Data columns (total 15 columns):
Lemma                     1449 non-null object
Word_Type                 1449 non-null object
Priority                  798 non-null object
Category                  1449 non-null object
Primary_subcategory       1449 non-null object
Secondary_subcategory     347 non-null object
Alt_Field                 479 non-null object
Alt_Sg_Replacement        711 non-null object
Alt_Sg_people_first       158 non-null object
Alt_Sg_reframed           298 non-null object
Alt_Pl_collective_noun    47 non-null object
Alt_Pl                    139 non-null object
Alt_Pl_people_first       28 non-null object
Alt_Pl_reframed           4 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 181.1+ KB


## Alternatives Single

### Single click+rephrase. Important order of columns with alternatives

In [339]:
#order for single form of alternatives provided by Tracey
df_rules["Sg_all"]= df_rules.loc[:, ("Alt_Sg_Replacement", "Alt_Field", "Alt_Pl_collective_noun", "Alt_Pl", "Alt_Sg_people_first", "Alt_Sg_reframed")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [340]:
# split alternatives in the row by " | " applying function split_form to convert string alternatives into array
df_rules["Sg_all_split"]=df_rules["Sg_all"].map(split_form)

## Alternatives Plural

### Single click + rephrase. Important order of columns with alternatives

In [341]:
#order for plural form of alternatives provided by Tracey
df_rules["Pl_all"]= df_rules.loc[:, ("Alt_Pl", "Alt_Pl_collective_noun", "Alt_Field", "Alt_Pl_people_first")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [342]:
# split alternatives in the row by " | " applying function split_form
df_rules["Pl_all_split"]=df_rules["Pl_all"].map(split_form)

### Merge alternatives into one column

In [343]:
source_col_loc = df_rules.columns.get_loc('Alt_Field') # column position starts from 0

In [344]:
#merge alternatives in one column by sign  " | "
df_rules['alternatives']= df_rules.iloc[:,source_col_loc:source_col_loc+7].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [345]:
# split alternatives in the row by " | " applying function split_form
df_rules["Alt_split"]=df_rules["alternatives"].map(split_form)

In [346]:
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,...,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives,Sg_all,Sg_all_split,Pl_all,Pl_all_split,alternatives,Alt_split
0,ADD,abbr,11-02-2022,style,abbreviation,NaN,NaN,attention deficit disorder,person with ADD | person with attention defici...,easily bored | inattentive | easily distracted...,...,NaN,NaN,NaN,NaN,attention deficit disorder | person with ADD |...,"[attention deficit disorder, person with ADD, ...",,[],attention deficit disorder | person with ADD |...,"[attention deficit disorder, person with ADD, ..."
1,ADHD,abbr,11-02-2022,style,abbreviation,NaN,NaN,attention deficit hyperactivity disorder,person with ADHD | person with attention defic...,fidgety | restless | easily bored,...,NaN,NaN,NaN,NaN,attention deficit hyperactivity disorder | per...,"[attention deficit hyperactivity disorder, per...",,[],attention deficit hyperactivity disorder | per...,"[attention deficit hyperactivity disorder, per..."
2,ASD,abbr,11-02-2022,style,abbreviation,NaN,autism spectrum disorder,neurodiversity,person with ASD | person with autism sprectru...,neurodevelopmental disorder characterized by d...,...,NaN,NaN,NaN,NaN,neurodiversity | autism spectrum disorder | pe...,"[neurodiversity, autism spectrum disorder, per...",autism spectrum disorder,[autism spectrum disorder],autism spectrum disorder | neurodiversity | pe...,"[autism spectrum disorder, neurodiversity, per..."
3,ASL,abbr,11-02-2022,style,abbreviation,NaN,NaN,American Sign Language,NaN,NaN,...,NaN,NaN,NaN,NaN,American Sign Language,[American Sign Language],,[],American Sign Language,[American Sign Language]
4,African queen,s,NaN,openly_discriminating,homophobia,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-,[-],-,[-],-,[-]


### Create subsets of dataframes

In [347]:
df_rules = df_rules[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"]]

### Check duplicates&homonyms

In [349]:
df_rules[df_rules["Lemma"].duplicated(keep = False)]

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
268,challenge,v,unconscious_bias,agentic,"[question, cross-examine, re-evaluate, object ...",,[],"[question, cross-examine, re-evaluate, object ..."
269,challenge,s,unconscious_bias,agentic,"[test, burning question, complex situation, ta...",complex questions | difficult tasks | burning ...,"[complex questions, difficult tasks, burning q...","[test, burning question, complex situation, ta..."
407,direct,v,unconscious_bias,agentic,"[guide, show, point, organize, encourage, assi...",,[],"[guide, show, point, organize, encourage, assi..."
408,direct,a,unconscious_bias,agentic,"[straightfoward, open, candid]",,[],"[straightfoward, open, candid]"
448,drive,v,unconscious_bias,agentic,"[invest in, support, work toward, take respons...",,[],"[invest in, support, work toward, take respons..."
449,drive,s,unconscious_bias,agentic,"[spirit, energy, motivation, appetite, growth ...",,[],"[spirit, energy, motivation, appetite, growth ..."
461,elderly,a,unconscious_bias,age_old,[older],,[],[older]
462,elderly,s,unconscious_bias,age_old,"[older adult, older people]",older people,[older people],"[older adult, older people]"
521,female,s,unconscious_bias,gender_identity,"[person, adult, woman, you, they, team, crowd,...",team | crowd,"[team, crowd]","[person, adult, woman, you, they, person who s..."
522,female,a,unconscious_bias,gender_identity,"[--- only if gender identity is relevant, --- ...",--- only if gender identity is relevant | --- ...,"[--- only if gender identity is relevant, --- ...","[--- only if gender identity is relevant, --- ..."


### Homonyms

In [350]:
# check how many homonyms
df_rules["Lemma"].duplicated(keep = False).sum()

46

In [351]:
df_homonyms = df_rules[df_rules["Lemma"].duplicated(keep = False)]

In [352]:
#check first 5 rows
df_homonyms.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
268,challenge,v,unconscious_bias,agentic,"[question, cross-examine, re-evaluate, object ...",,[],"[question, cross-examine, re-evaluate, object ..."
269,challenge,s,unconscious_bias,agentic,"[test, burning question, complex situation, ta...",complex questions | difficult tasks | burning ...,"[complex questions, difficult tasks, burning q...","[test, burning question, complex situation, ta..."
407,direct,v,unconscious_bias,agentic,"[guide, show, point, organize, encourage, assi...",,[],"[guide, show, point, organize, encourage, assi..."
408,direct,a,unconscious_bias,agentic,"[straightfoward, open, candid]",,[],"[straightfoward, open, candid]"
448,drive,v,unconscious_bias,agentic,"[invest in, support, work toward, take respons...",,[],"[invest in, support, work toward, take respons..."


In [353]:
#create subset of dataframe to drop unnecessary columns
df_homonyms = df_rules[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [354]:
#save homonyms to csv file after preprocessing
df_homonyms.to_csv("homonyms_words_EN.csv", index=False)

### Remove homonyms from dataframe for the next steps pre-processing

In [355]:
homonyms_list = list(df_rules[df_rules["Lemma"].duplicated()]["Lemma"])

In [356]:
df_rules = df_rules.query("Lemma != @homonyms_list")

### Acronyms (abbreviations)

In [362]:
#check word_type
df_rules["Word_Type"].unique()

array(['abbr', 's', 'i', 'a+s', 'a', 'v', 'v+a+s', 'v+s', 'adv+a', 'v+a',
       'a+a+s', 's,v', 'adv', 'v, a', 's+s'], dtype=object)

In [363]:
df_abbr = df_rules[df_rules["Word_Type"]=='abbr']

In [364]:
#create subset of dataframe to drop unnecessary columns
df_abbr = df_abbr[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [365]:
#save acronyms (abbreviations) to csv file after preprocessing
df_abbr.to_csv("abbreviations_EN.csv", index=False)

### Remove acronymes (abbr) from dataframe for the next steps pre-processing

In [366]:
#remove abbr
abbr_list = list(df_abbr["Lemma"])

In [367]:
df_rules = df_rules.query("Lemma != @abbr_list")

### Openly_discriminating

In [368]:
df_openly_discrim = df_rules[df_rules["Category"]=="openly_discriminating"]

In [369]:
#info open_discrim
df_openly_discrim.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 338 entries, 4 to 1444
Data columns (total 8 columns):
Lemma                  338 non-null object
Word_Type              338 non-null object
Category               338 non-null object
Primary_subcategory    338 non-null object
Sg_all_split           338 non-null object
Pl_all                 338 non-null object
Pl_all_split           338 non-null object
Alt_split              338 non-null object
dtypes: object(8)
memory usage: 23.8+ KB


In [370]:
df_openly_discrim.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
4,African queen,s,openly_discriminating,homophobia,[-],-,[-],[-]
6,Aunt Jemima,s,openly_discriminating,racism,[-],-,[-],[-]
7,Barbie,s,openly_discriminating,racism,[-],-,[-],[-]
8,Beastie Boy,s,openly_discriminating,antisemitism,[-],-,[-],[-]
9,Beulah,s,openly_discriminating,racism,[-],-,[-],[-]


In [371]:
#split dataframe into two words and sentences using function split_words_sentences
df_open_dis_word, df_open_dis_sentence = split_words_sentences(df_openly_discrim)

In [372]:
df_open_dis_word.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
0,Barbie,s,openly_discriminating,racism,[-],-,[-],[-]
1,Beulah,s,openly_discriminating,racism,[-],-,[-],[-]
2,Bin-Laden,s,openly_discriminating,antimuslim,[-],-,[-],[-]
3,Binladen,s,openly_discriminating,antimuslim,[-],-,[-],[-]
4,Brownie,s,openly_discriminating,homophobia,[-],-,[-],[-]


In [373]:
#create subset of dataframe to drop unnecessary columns
df_open_dis_word = df_open_dis_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
df_open_dis_sentence = df_open_dis_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [374]:
#save to csv file after preprocessing
df_open_dis_word.to_csv("open_dis_words_EN.csv", index=False)

In [375]:
#save to csv file after preprocessing
df_open_dis_sentence.to_csv("open_dis_sentences_EN.csv", index=False)

### Style

In [376]:
df_style = df_rules[df_rules["Category"]=="style"]

In [377]:
#info style
df_style.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 130 entries, 42 to 1441
Data columns (total 8 columns):
Lemma                  130 non-null object
Word_Type              130 non-null object
Category               130 non-null object
Primary_subcategory    130 non-null object
Sg_all_split           130 non-null object
Pl_all                 130 non-null object
Pl_all_split           130 non-null object
Alt_split              130 non-null object
dtypes: object(8)
memory usage: 9.1+ KB


In [378]:
#split dataframe into two words and sentences using function split_words_sentences
df_style_word, df_style_sentence = split_words_sentences(df_style)

In [379]:
#create subset of dataframe to drop unnecessary columns
df_style_word = df_style_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
df_style_sentence = df_style_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [380]:
#save to csv file after preprocessing
df_style_word.to_csv("style_words_EN.csv", index=False)

In [381]:
#save to csv file after preprocessing
df_style_sentence.to_csv("style_sentences_EN.csv", index=False)

### Inclusive

In [382]:
df_inclusive = df_rules[df_rules["Category"]=="inclusive"]

In [383]:
#info inclusive
df_inclusive.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 186 entries, 68 to 1445
Data columns (total 8 columns):
Lemma                  186 non-null object
Word_Type              186 non-null object
Category               186 non-null object
Primary_subcategory    186 non-null object
Sg_all_split           186 non-null object
Pl_all                 186 non-null object
Pl_all_split           186 non-null object
Alt_split              186 non-null object
dtypes: object(8)
memory usage: 13.1+ KB


In [384]:
df_inclusive.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
68,accessibility,s,inclusive,d_and_i,[],,[],[]
75,affectionate,a,inclusive,communal,[],,[],[]
76,affinity group,s,inclusive,d_and_i,[],,[],[]
82,ages,s,inclusive,d_and_i,[],,[],[]
85,agree,v,inclusive,communal,[],,[],[]


In [385]:
#split dataframe into two words and sentences using function split_words_sentences
df_inclusive_word, df_inclusive_sentence = split_words_sentences(df_inclusive)

In [386]:
#create subset of dataframe to drop unnecessary columns
df_inclusive_word = df_inclusive_word[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]
df_inclusive_sentence = df_inclusive_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]

In [387]:
df_inclusive_sentence.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 4 columns):
Lemma                  24 non-null object
Word_Type              24 non-null object
Category               24 non-null object
Primary_subcategory    24 non-null object
dtypes: object(4)
memory usage: 896.0+ bytes


In [388]:
#save to csv file after preprocessing
df_inclusive_word.to_csv("inclusive_words_EN.csv", index=False)

In [389]:
#save to csv file after preprocessing
df_inclusive_sentence.to_csv("inclusive_sentences_EN.csv", index=False)

### Gendered

In [391]:
df_gendered = df_rules[df_rules["Category"]=="gendered"]

In [392]:
#info gendered
df_gendered.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 180 entries, 20 to 1417
Data columns (total 8 columns):
Lemma                  180 non-null object
Word_Type              180 non-null object
Category               180 non-null object
Primary_subcategory    180 non-null object
Sg_all_split           180 non-null object
Pl_all                 180 non-null object
Pl_all_split           180 non-null object
Alt_split              180 non-null object
dtypes: object(8)
memory usage: 12.7+ KB


In [393]:
#split dataframe into two words and sentences using function split_words_sentences
df_gendered_word, df_gendered_sentence = split_words_sentences_sing_plur(df_gendered)

In [394]:
df_gendered_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 8 columns):
Lemma                  128 non-null object
Word_Type              128 non-null object
Category               128 non-null object
Primary_subcategory    128 non-null object
Sg_all_split           128 non-null object
Pl_all                 128 non-null object
Pl_all_split           128 non-null object
Alt_split              128 non-null object
dtypes: object(8)
memory usage: 8.1+ KB


In [395]:
df_gendered_word_noun = df_gendered_word.query("Word_Type == 's'")

In [396]:
df_gendered_word_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 114 entries, 0 to 126
Data columns (total 8 columns):
Lemma                  114 non-null object
Word_Type              114 non-null object
Category               114 non-null object
Primary_subcategory    114 non-null object
Sg_all_split           114 non-null object
Pl_all                 114 non-null object
Pl_all_split           114 non-null object
Alt_split              114 non-null object
dtypes: object(8)
memory usage: 8.0+ KB


In [397]:
df_gendered_word_no_noun = df_gendered_word.query("Word_Type != 's'")

In [398]:
#drop unnecessary column "Pl_all"
df_gendered_word_noun = df_gendered_word_noun.drop(labels=["Pl_all"], axis=1)
df_gendered_word_no_noun = df_gendered_word_no_noun.drop(labels=["Pl_all"], axis=1)
df_gendered_sentence = df_gendered_sentence.drop(labels=["Pl_all"], axis=1)

In [399]:
#save to csv file after preprocessing
df_gendered_word_noun.to_csv("gendered_noun_words_EN.csv", index=False)

In [400]:
#save to csv file after preprocessing
df_gendered_word_no_noun.to_csv("gendered_no_noun_words_EN.csv", index=False)

In [401]:
df_gendered_sentence.to_csv("gendered_sentences_EN.csv", index=False)

### Unconscious Bias

In [402]:
df_ub = df_rules_sing_plur[df_rules_sing_plur["Category"]=='unconscious_bias']

In [403]:
#info unc_bias
df_ub.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 552 entries, 5 to 1449
Data columns (total 8 columns):
Lemma                  552 non-null object
Word_Type              552 non-null object
Category               552 non-null object
Primary_subcategory    552 non-null object
Sg_all_split           552 non-null object
Pl_all                 552 non-null object
Pl_all_split           552 non-null object
Alt_split              552 non-null object
dtypes: object(8)
memory usage: 38.8+ KB


In [404]:
#split dataframe into two words and sentences using function split_words_sentences
df_ub_word, df_ub_sentence = split_words_sentences(df_ub)

In [405]:
df_ub_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 8 columns):
Lemma                  338 non-null object
Word_Type              338 non-null object
Category               338 non-null object
Primary_subcategory    338 non-null object
Sg_all_split           338 non-null object
Pl_all                 338 non-null object
Pl_all_split           338 non-null object
Alt_split              338 non-null object
dtypes: object(8)
memory usage: 21.2+ KB


In [406]:
#filter datframe to have plural for subj(noun) without special caracters and empty strings
df_ub_word_pl = df_ub_word[(df_ub_word["Pl_all"]!='')&(df_ub_word["Pl_all"]!='-')&(df_ub_word["Word_Type"]=='s')]

In [407]:
df_ub_word_pl.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 30 entries, 0 to 329
Data columns (total 8 columns):
Lemma                  30 non-null object
Word_Type              30 non-null object
Category               30 non-null object
Primary_subcategory    30 non-null object
Sg_all_split           30 non-null object
Pl_all                 30 non-null object
Pl_all_split           30 non-null object
Alt_split              30 non-null object
dtypes: object(8)
memory usage: 2.1+ KB


#### save the left data into separate dataframe

In [408]:
plural_list = list(df_ub_word_pl["Lemma"])

In [409]:
df_ub_word_left = df_ub_word.query("Lemma != @plural_list")

#### remove unnecessary column "Pl_all" 

In [410]:
df_ub_word_left = df_ub_word_left.drop(labels=["Pl_all"], axis=1)
df_ub_word_pl = df_ub_word_pl.drop(labels=["Pl_all"], axis=1)
df_ub_sentence = df_ub_sentence.drop(labels=["Pl_all"], axis=1)

#### save to csv files after preprocessing

In [411]:
#save to csv files after preprocessing
df_ub_word_left.to_csv("ub_no_plur_words_EN.csv", index=False)
df_ub_word_pl.to_csv("ub_plur_words_EN.csv", index=False)
df_ub_sentence.to_csv("ub_sentences_EN.csv", index=False)